In [1]:
from lm_polygraph.model_adapters.togetherai_adapter import TogetherAIAdapter
import os

os.environ["TOGETHER_API_KEY"] = "MY_TOKEN"

adapter = TogetherAIAdapter()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MyModel:
    def __init__(self, model_path: str = "google/gemma-3n-E4B-it"):
        self.model_path = model_path
    
    def prepare_input(self, prompt):
        return [
            {"role": "user", "content": prompt},
        ]

In [3]:
from lm_polygraph.stat_calculators.graybox_stats import GrayBoxStatsCalculator
from lm_polygraph.stat_calculators import GreedyAlternativesNLICalculator
from lm_polygraph.estimators import ClaimConditionedProbability
from lm_polygraph.utils.deberta import Deberta

nli_model = Deberta()

model = MyModel()
stat_calculators = [
    GrayBoxStatsCalculator(),
    GreedyAlternativesNLICalculator(nli_model),
]
estimator = ClaimConditionedProbability()

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
deps = {'model': model}
texts = ['2+2=?']
for stat_calculator in stat_calculators:
    deps.update(stat_calculator(deps, texts, adapter))
estimator(deps)

array([-0.99399466])